# What "Learning from Data" Actually Means — Before Any Model

Companion notebook for Act 1, Article 1 of the ML series.  
Runs standalone — no dependencies on other folders.

Article link: *(add when published)*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Pinned URLs — same data the article used
DATA_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
ZONE_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi+_zone_lookup.csv"

## Getting the Data

In [ ]:
df = pd.read_parquet(DATA_URL)

print(df.shape)
print(df.dtypes)
print(df.head(3))

## What the Raw Data Actually Looks Like

In [ ]:
cols = ["passenger_count", "trip_distance", "fare_amount", "tip_amount", "total_amount"]
print(df[cols].describe())

In [ ]:
# Missing values — notice the identical percentage across five columns
missing = df.isnull().sum() / len(df) * 100
print(missing[missing > 0].sort_values(ascending=False))

## Cleaning With Intent, Not Habit

In [ ]:
def clean_trips(df):
    df = df[df["fare_amount"] > 0]
    df = df[df["tip_amount"] >= 0]
    df = df[df["passenger_count"] >= 1]
    df = df[df["passenger_count"] <= 5]
    df = df[df["trip_distance"] > 0]
    df = df[df["trip_distance"] <= 50]
    df = df[df["fare_amount"] >= 3.0]
    df = df[df["fare_amount"] <= 200]

    df["trip_duration_min"] = (
        df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
    ).dt.total_seconds() / 60

    df = df[df["trip_duration_min"] > 1]
    df = df[df["trip_duration_min"] <= 120]
    df = df[df["RatecodeID"].isin([1, 2])]
    return df.reset_index(drop=True)


df_clean = clean_trips(df)

print(f"Original rows : {len(df):,}")
print(f"After cleaning: {len(df_clean):,}")
print(f"Removed       : {len(df) - len(df_clean):,} ({(1 - len(df_clean)/len(df))*100:.1f}%)")

## Feature Engineering — Where the Real Work Is

In [ ]:
def engineer_features(df):
    dt = df["tpep_pickup_datetime"]

    df["pickup_hour"]      = dt.dt.hour
    df["pickup_dayofweek"] = dt.dt.dayofweek
    df["pickup_month"]     = dt.dt.month
    df["is_weekend"]       = (dt.dt.dayofweek >= 5).astype(int)
    df["is_rush_hour"]     = (
        ((dt.dt.hour >= 7)  & (dt.dt.hour <= 9)) |
        ((dt.dt.hour >= 16) & (dt.dt.hour <= 19))
    ).astype(int)
    df["is_late_night"]    = ((dt.dt.hour >= 22) | (dt.dt.hour <= 5)).astype(int)

    df["avg_speed_mph"] = (
        df["trip_distance"] / (df["trip_duration_min"] / 60)
    ).clip(upper=80)
    df["fare_per_mile"]    = (df["fare_amount"] / df["trip_distance"]).clip(upper=50)
    df["is_card_payment"]  = (df["payment_type"] == 1).astype(int)
    df["is_jfk_rate"]      = (df["RatecodeID"] == 2).astype(int)

    return df


df_feat = engineer_features(df_clean)

## The Cash Tip Trap

In [ ]:
cash_trips = df_feat[df_feat["payment_type"] == 2]
card_trips  = df_feat[df_feat["payment_type"] == 1]

print(f"Cash trips with tip_amount > 0 : {(cash_trips['tip_amount'] > 0).mean():.2%}")
print(f"Card trips with tip_amount > 0 : {(card_trips['tip_amount'] > 0).mean():.2%}")

# Filter to card payments only
df_model = df_feat[df_feat["is_card_payment"] == 1].copy()
print(f"\nCard payment trips: {len(df_model):,}")

## What the Distributions Are Telling You

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_model["tip_amount"], bins=100, color="#2c3e50", edgecolor="none")
axes[0].set_title("Tip Amount — Raw")
axes[0].set_xlabel("Tip ($)")
axes[0].set_ylabel("Count")

df_model["tip_pct"] = df_model["tip_amount"] / df_model["fare_amount"] * 100
axes[1].hist(df_model["tip_pct"].clip(0, 50), bins=100, color="#2c3e50", edgecolor="none")
axes[1].set_title("Tip as % of Fare")
axes[1].set_xlabel("Tip (%)")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
# The 20% spike
at_20 = ((df_model["tip_pct"] >= 19.5) & (df_model["tip_pct"] <= 20.5)).sum()
print(f"Trips tipping exactly ~20%: {at_20:,} ({at_20/len(df_model):.1%})")

## Correlation — and Why Changing the Target Changes Everything

In [ ]:
features = [
    "trip_distance", "fare_amount", "pickup_hour",
    "is_weekend", "is_rush_hour", "is_late_night",
    "avg_speed_mph", "fare_per_mile", "trip_duration_min",
    "passenger_count", "is_jfk_rate",
]

print("── Correlation with tip_amount ──")
corr_amt = df_model[features + ["tip_amount"]].corr()["tip_amount"].drop("tip_amount")
print(corr_amt.sort_values(ascending=False))

print("\n── Correlation with tip_pct ──")
corr_pct = df_model[features + ["tip_pct"]].corr()["tip_pct"].drop("tip_pct")
print(corr_pct.sort_values(ascending=False))

## Preparing the Final Feature Set

In [ ]:
zones    = pd.read_csv(ZONE_URL)
zone_map = zones.set_index("LocationID")["Borough"].to_dict()
df_model["pickup_borough"]  = df_model["PULocationID"].map(zone_map)
df_model["dropoff_borough"] = df_model["DOLocationID"].map(zone_map)
borough_enc = {b: i for i, b in enumerate(df_model["pickup_borough"].dropna().unique())}
df_model["pickup_borough_enc"]  = df_model["pickup_borough"].map(borough_enc)
df_model["dropoff_borough_enc"] = df_model["dropoff_borough"].map(borough_enc)

MODEL_FEATURES = [
    "trip_distance", "trip_duration_min", "fare_amount",
    "pickup_hour", "pickup_dayofweek", "is_weekend",
    "is_rush_hour", "is_late_night", "avg_speed_mph",
    "fare_per_mile", "passenger_count", "is_jfk_rate",
    "pickup_borough_enc", "dropoff_borough_enc",
]
TARGET = "tip_pct"

df_final = df_model[MODEL_FEATURES + [TARGET, "tpep_pickup_datetime"]].dropna()

print(f"Final dataset shape: {df_final[MODEL_FEATURES + [TARGET]].shape}")
print(df_final[TARGET].describe())

## Train/Test Split — Time-Based

In [ ]:
df_final = df_final.sort_values("tpep_pickup_datetime")

# Last 25% of January as test set
cutoff     = df_final["tpep_pickup_datetime"].quantile(0.75)
train_mask = df_final["tpep_pickup_datetime"] <= cutoff

X_train = df_final.loc[ train_mask, MODEL_FEATURES]
X_test  = df_final.loc[~train_mask, MODEL_FEATURES]
y_train = df_final.loc[ train_mask, TARGET]
y_test  = df_final.loc[~train_mask, TARGET]

print(f"Training samples : {len(X_train):,}")
print(f"Test samples     : {len(X_test):,}")
print("\nX_train / y_train / X_test / y_test ready.")
print("No model.fit() — that is the next article.")